# Calidad de datos

## Imports

In [11]:
import pandas as pd
import sqlite3
import json


### Variables globales de conexión

In [12]:
DB_NAME = "openbeauty"
DB_PATH = f"../data/{DB_NAME}.db"
TABLE_NAME = "raw_products"

conn = sqlite3.connect(DB_PATH)


Objetivo: Responder que tan util es esta data ?
* Unicidad
* Concistencia
* Inconsistencia 

In [13]:
df_raw_products = pd.read_sql("SELECT * FROM raw_products", conn)

In [14]:
df_raw_products.head()

,_id,code,rev,update_key,brands,product_name,product_type,countries,countries_tags,countries_hierarchy,...,created_t,last_modified_t,last_updated_t,creator,last_editor,last_modified_by,data_quality_tags,page,batch_id,dtinserted
0,8410757001090,8410757001090,34,key_1743168287,S'nonas,Crema Manos,beauty,"Morocco, United States, en:france","[""en:france"", ""en:morocco"", ""en:united-states""]","[""en:france"", ""en:morocco"", ""en:united-states""]",...,1731329832,1779248127,1779248127,smoothie-app,agenticcommonsbot,agenticcommonsbot,"[""en:no-packaging-data""]",1,1,2026-05-22 16:05:44.417078
1,6281006408647,6281006408647,30,brands,"Unilever,Vaseline",,beauty,"Germany, Morocco, en:saudi-arabia","[""en:germany"", ""en:morocco"", ""en:saudi-arabia""]","[""en:germany"", ""en:morocco"", ""en:saudi-arabia""]",...,1731340361,1771266702,1771266702,smoothie-app,scanbot,scanbot,"[""en:no-packaging-data""]",1,1,2026-05-22 16:05:44.417078
2,80466468,80466468,68,brands,"unilever, dove",DOVE Déodorant Femme Anti-Transpirant Stick Or...,beauty,"France, Libya, Morocco, Saudi Arabia, Spain, e...","[""en:algeria"", ""en:france"", ""en:libya"", ""en:mo...","[""en:algeria"", ""en:france"", ""en:libya"", ""en:mo...",...,1537085263,1779261226,1779261226,openfoodfacts-contributors,agenticcommonsbot,agenticcommonsbot,"[""en:packaging-data-incomplete""]",1,1,2026-05-22 16:05:44.417078
3,3337875598996,3337875598996,33,brands,CeraVe,moisturising cream,beauty,"France,Morocco,Romania","[""en:france"", ""en:morocco"", ""en:romania""]","[""en:france"", ""en:morocco"", ""en:romania""]",...,1730480414,1779263998,1779263998,smoothie-app,agenticcommonsbot,agenticcommonsbot,"[""en:packaging-data-incomplete""]",1,1,2026-05-22 16:05:44.417078
4,3574669909594,3574669909594,35,key_1743168287,Johnson’s baby,,beauty,"Algeria, Morocco, en:ukraine","[""en:algeria"", ""en:morocco"", ""en:ukraine""]","[""en:algeria"", ""en:morocco"", ""en:ukraine""]",...,1660916439,1775820851,1775820851,smoothie-app,foodless,foodless,"[""en:no-packaging-data""]",1,1,2026-05-22 16:05:44.417078


In [15]:
df_raw_products.shape

(50000, 32)

In [16]:
df = df_raw_products

## Data Quality

### Completitud 

In [45]:
df[['countries','countries_tags', 'countries_hierarchy']].isnull().sum()

countries              736
countries_tags         495
countries_hierarchy    736
dtype: int64

Observando que la cantidad de nulos dentro de las variables countries, se verificara sus diferencias para unificarlas

In [46]:
df[['countries','countries_tags', 'countries_hierarchy']].head()

,countries,countries_tags,countries_hierarchy
0,"Morocco, United States, en:france","[""en:france"", ""en:morocco"", ""en:united-states""]","[""en:france"", ""en:morocco"", ""en:united-states""]"
1,"Germany, Morocco, en:saudi-arabia","[""en:germany"", ""en:morocco"", ""en:saudi-arabia""]","[""en:germany"", ""en:morocco"", ""en:saudi-arabia""]"
2,"France, Libya, Morocco, Saudi Arabia, Spain, e...","[""en:algeria"", ""en:france"", ""en:libya"", ""en:mo...","[""en:algeria"", ""en:france"", ""en:libya"", ""en:mo..."
3,"France,Morocco,Romania","[""en:france"", ""en:morocco"", ""en:romania""]","[""en:france"", ""en:morocco"", ""en:romania""]"
4,"Algeria, Morocco, en:ukraine","[""en:algeria"", ""en:morocco"", ""en:ukraine""]","[""en:algeria"", ""en:morocco"", ""en:ukraine""]"


In [49]:
df[['categories', 'categories_hierarchy']].head()

,categories,categories_hierarchy
0,"Creams,Non-food-products,Open-beauty-facts","[""en:Creams"", ""en:Non-food-products"", ""en:Open..."
1,"Incorrect product type,Non-food-products,Open-...","[""en:incorrect-product-type"", ""en:Non-food-pro..."
2,"Déodorants anti-transpirants, en:open-beauty-f...","[""en:D\u00e9odorants anti-transpirants"", ""en:o..."
3,fr:baume hydratant,"[""fr:baume hydratant""]"
4,,[]


In [50]:
df[['categories','categories_hierarchy']].isnull().sum()

categories              18086
categories_hierarchy    18056
dtype: int64

#### Unificando variables que son "iguales"

In [52]:
df['canonical_countries'] = df['countries_hierarchy'].combine_first(df['countries_tags']).combine_first(df['countries'])

In [53]:
df['canonical_categories'] = df['categories_hierarchy'].combine_first(df['categories'])

In [54]:
df.columns

Index(['_id', 'code', 'rev', 'update_key', 'brands', 'product_name',
       'product_type', 'countries', 'countries_tags', 'countries_hierarchy',
       'categories_hierarchy', 'categories', 'product_quantity',
       'product_quantity_unit', 'quantity', 'ingredients_n',
       'known_ingredients_n', 'unknown_ingredients_n', 'completeness',
       'scans_n', 'unique_scans_n', 'popularity_tags', 'created_t',
       'last_modified_t', 'last_updated_t', 'creator', 'last_editor',
       'last_modified_by', 'data_quality_tags', 'page', 'batch_id',
       'dtinserted', 'canonical_countries', 'canonical_categories'],
      dtype='str')

Definiendo las columnas que determinan la calidad de los datos

In [56]:
quality_columns = [
    "code",
    "product_name",
    "product_type",
    "brands",
    "canonical_categories",
    "canonical_countries",
    "ingredients_n"
]

Definiendo las columnas criticas que determinan la calidad 

In [ ]:
critical_columns = [
    "product_name",
    "brands",
    "canonical_categories",
    "canonical_countries"
]